# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All dataset entities—including record sets, fields, and columns—are referenced by their unique `@id` as recommended in FAIR data best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
# Retrieve metadata as a JSON object for display
metadata = dataset.metadata.to_json()

print(f"Dataset name: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`). This helps identify entities for further extraction and exploration.

In [ ]:
# List available record sets and their field IDs. We'll access entities by their `@id` only.

print("Available Record Sets and Their Fields:")
for record_set in dataset.record_sets:
    rs_json = record_set.to_json()
    print(f"• Record set @id: {rs_json['@id']}")
    print(f"  Name: {rs_json.get('name', rs_json.get('@id'))}")
    print(f"  Description: {rs_json.get('description', 'No description')}")
    # List fields in this record set
    if 'field' in rs_json:
        fields = rs_json['field']
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields and their @id:")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - {field.get('@id', 'Unknown')}: {field.get('name', 'Unnamed Field')}")
                elif isinstance(field, str):
                    print(f"    - {field}")
    print()

In [ ]:
# OPTIONAL: Display a sample record (by @id) for a chosen record set
# Replace <RECORD_SET_ID> below with an actual @id from the cell above after inspection.

# Example (to be replaced with a valid @id):
# for rec in dataset.records(record_set='@id_of_record_set'):
#     print(rec)
#     break  # Just show one record

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis using the record set and field `@id`s discovered above.

In [ ]:
# List all available record set @id's for extraction
record_set_ids = [rs.to_json()['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set @id: {record_set_id}, shape: {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set @id: {record_set_id}")

# If one or more DataFrames loaded, show the columns and sample from the first one
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set @id '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head(3))

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. All operations reference fields by their canonical `@id`.

In [ ]:
# We'll attempt EDA on the first available record set as an example.
import numpy as np

if dataframes:
    record_set_id = list(dataframes.keys())[0]  # Choose first loaded record set
    df = dataframes[record_set_id]
    print(f"EDA for record set @id: {record_set_id}")

    # Try to identify a numeric field by @id, e.g., 'log_likelihood', 'age', etc.
    # List the columns for reference
    print("Available fields (by @id):", df.columns.tolist())

    # Try to find a numeric column (e.g., one with float/int values and not too many missing)
    candidate_numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Using numeric field (by @id): {numeric_field}")
    else:
        print('No numeric field found; please review field @id's manually from above.')
        numeric_field = None

    if numeric_field:
        # Filtering: e.g., keep only values above 0
        threshold = 0
        mask = df[numeric_field] > threshold
        filtered_df = df[mask].copy()
        print(f"Filtered records with {numeric_field} > {threshold}: {filtered_df.shape[0]} rows")

        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field (categorical):
        candidate_group_fields = [col for col in df.columns if col != numeric_field and df[col].nunique() < 10]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping by field (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df)
        else:
            print('No suitable group field found.')

## 5. Visualization
Visualize the distribution of a numeric field or the relationship between two fields based on their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, referencing all entities by their canonical `@id`. We reviewed available record sets and fields, loaded data, performed filtering and normalization on a numeric field, and visualized results accordingly. For detailed field and analysis selection, please refer to the printed `@id`s above and consult the dataset schema for context.

Further exploration is encouraged, using this notebook as a starting template for FAIR data processing.